In [2]:
# ============================================================
# 0. Import dan konfigurasi
# ============================================================
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

DATA_DIR = "."                      # folder tempat train.csv dan test.csv berada
TARGET = "satisfaction"
POSITIVE_LABEL = "satisfied"        # kelas yang dihitung sebagai positif (1)
ID_COL = "id"
ORDINAL_COL = "Class"
NOMINAL_COLS = ["Gender", "Customer Type", "Type of Travel"]
DELAY_COL = "Arrival Delay in Minutes"
N_SPLITS = 5
SEED = 42

# ============================================================
# 1. Load data
# ============================================================
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
print("Train:", train.shape, "| Test:", test.shape)

# ============================================================
# 2. Cek nilai target
# ============================================================
print(train[TARGET].dtype)
print(train[TARGET].value_counts())

assert train[TARGET].dtype == bool, f"Tipe target tidak terduga: {train[TARGET].dtype}"
assert set(train[TARGET].unique()) == {True, False}, "Target bukan biner"

rating_cols = ["Online boarding", "Seat comfort", "Inflight wifi service", "Cleanliness"]
print(train.groupby(TARGET)[rating_cols].mean().round(2))

# ============================================================
# 3. Imputasi Arrival Delay (median dihitung dari TRAIN saja)
# ============================================================
median_delay = train[DELAY_COL].median()
train[DELAY_COL] = train[DELAY_COL].fillna(median_delay).astype(int)
test[DELAY_COL] = test[DELAY_COL].fillna(median_delay).astype(int)

# ============================================================
# 4. Target: satisfied = 1, selainnya = 0
# ============================================================
y = train[TARGET].astype(int)   # True -> 1, False -> 0 (sesuaikan jika verifikasi menunjukkan sebaliknya)

assert y.nunique() == 2, "y hanya punya satu kelas, cek preprocessing target"
print("Distribusi y:", y.value_counts().to_dict())



# ============================================================
# 5. Pisahkan fitur dan simpan id test untuk submission
# ============================================================
test_ids = test[ID_COL].copy()
train_feat = train.drop(columns=[TARGET, ID_COL])
test_feat = test.drop(columns=[ID_COL, TARGET], errors="ignore")

# ============================================================
# 6. Label encoding untuk Class
#    fit di gabungan train+test (hanya fitur, tidak menyentuh target)
# ============================================================
le_class = LabelEncoder()
le_class.fit(pd.concat([train_feat[ORDINAL_COL], test_feat[ORDINAL_COL]]).astype(str))
train_feat[ORDINAL_COL] = le_class.transform(train_feat[ORDINAL_COL].astype(str))
test_feat[ORDINAL_COL] = le_class.transform(test_feat[ORDINAL_COL].astype(str))

# ============================================================
# 7. One-hot encoding pada gabungan train+test
# ============================================================
train_feat = train_feat.assign(is_train=1)
test_feat = test_feat.assign(is_train=0)
combined = pd.concat([train_feat, test_feat], axis=0, ignore_index=True)
combined = pd.get_dummies(combined, columns=NOMINAL_COLS, drop_first=True, dtype=int)

X = combined[combined["is_train"] == 1].drop(columns="is_train").reset_index(drop=True)
X_test = combined[combined["is_train"] == 0].drop(columns="is_train").reset_index(drop=True)

# ============================================================
# 8. Pengecekan sebelum training
# ============================================================
assert len(X) == len(y), "Jumlah baris X dan y harus sama"
assert list(X.columns) == list(X_test.columns), "Kolom X dan X_test harus sama"
assert all(d.kind in "biuf" for d in X.dtypes), "Ada kolom non-numerik di X"
print("X:", X.shape, "| X_test:", X_test.shape)

# ============================================================
# 9. Parameter model dan wadah hasil
# ============================================================
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

lgb_params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "random_state": SEED,
    "verbose": -1,
}

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
fold_scores = []

# ============================================================
# 10. Training K-fold
# ============================================================
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n==== Training fold {fold + 1} ====")

    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval = lgb.Dataset(X_va, label=y_va, reference=dtrain)

    model = lgb.train(
        lgb_params,
        dtrain,
        num_boost_round=3000,
        valid_sets=[dval],
        valid_names=["val"],
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(200)],
    )

    oof_preds[val_idx] = model.predict(X_va, num_iteration=model.best_iteration)
    test_preds += model.predict(X_test, num_iteration=model.best_iteration) / N_SPLITS

    fold_auc = roc_auc_score(y_va, oof_preds[val_idx])
    fold_scores.append(fold_auc)
    print(f"Fold {fold + 1} | AUC = {fold_auc:.5f} | best_iter = {model.best_iteration}")

# ============================================================
# 11. Evaluasi CV
# ============================================================
cv_auc = roc_auc_score(y, oof_preds)
print(f"\nCV AUC (OOF)       = {cv_auc:.5f}")
print(f"Rata-rata per fold = {np.mean(fold_scores):.5f} ± {np.std(fold_scores):.5f}")

# ============================================================
# 12. Buat file submission
# ============================================================
submission = pd.DataFrame({ID_COL: test_ids, TARGET: test_preds})

assert list(submission.columns) == list(sample_sub.columns), "Nama kolom submission tidak sesuai"
assert len(submission) == len(sample_sub), "Jumlah baris submission tidak sesuai"
assert submission[TARGET].between(0, 1).all(), "Prediksi harus berada di rentang 0 sampai 1"

submission.to_csv("submission.csv", index=False)
print(submission.head())

Train: (699635, 23) | Test: (299844, 22)
bool
satisfaction
False    389296
True     310339
Name: count, dtype: int64
              Online boarding  Seat comfort  Inflight wifi service  \
satisfaction                                                         
False                    2.72          3.11                   2.45   
True                     4.17          4.14                   3.20   

              Cleanliness  
satisfaction               
False                2.99  
True                 3.87  
Distribusi y: {0: 389296, 1: 310339}
X: (699635, 21) | X_test: (299844, 21)

==== Training fold 1 ====
[200]	val's auc: 0.957917
[400]	val's auc: 0.958369
[600]	val's auc: 0.958482
[800]	val's auc: 0.958617
[1000]	val's auc: 0.958641
[1200]	val's auc: 0.958695
Fold 1 | AUC = 0.95870 | best_iter = 1202

==== Training fold 2 ====
[200]	val's auc: 0.95681
[400]	val's auc: 0.957278
[600]	val's auc: 0.957427
[800]	val's auc: 0.957504
[1000]	val's auc: 0.957544
[1200]	val's auc: 0.957602
[14